# Estimating 2026 Fed rate-hike hazards from the Treasury yield curve

**Question.** Using Treasury yields at several maturities, what is the probability that the Federal Reserve raises its target rate in September, October, November, or December 2026?

**As-of date:** the notebook automatically uses the latest observation available from the Federal Reserve Economic Data (FRED) download service. To make a submitted result reproducible, set `AS_OF_DATE` to a fixed date.

This notebook is written for a reader without much financial background. It separates:

1. what the raw government data mean;
2. how the historical model is built;
3. what a hazard probability means;
4. how we convert conditional hazards into first-hike probabilities; and
5. how the model compares with professional market evidence.

> **Important limitation:** Treasury yields are not contracts on individual FOMC outcomes. Fed-funds futures are the professional instrument normally used for meeting-specific probabilities. The estimates below are model-based probabilities inferred from historical Treasury-curve patterns—not direct market-implied probabilities.

## 1. Trusted sources

All model inputs come from U.S. government or Federal Reserve sources:

- [Federal Reserve H.15 / FRED Treasury series](https://fred.stlouisfed.org/categories/115): constant-maturity Treasury yields. The underlying source is the Board of Governors of the Federal Reserve System.
- [DGS3MO](https://fred.stlouisfed.org/series/DGS3MO), [DGS6MO](https://fred.stlouisfed.org/series/DGS6MO), [DGS10](https://fred.stlouisfed.org/series/DGS10), and [DGS30](https://fred.stlouisfed.org/series/DGS30): 3-month, 6-month, 10-year, and 30-year yields.
- [DFEDTARU](https://fred.stlouisfed.org/series/DFEDTARU) and [DFEDTARL](https://fred.stlouisfed.org/series/DFEDTARL): upper and lower limits of the federal-funds target range.
- [Official 2026 FOMC calendar](https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm): September 15–16, October 27–28, and December 8–9. **There is no scheduled November meeting.**

FRED is operated by the Federal Reserve Bank of St. Louis. It distributes these official series in a stable CSV format and requires no API key for the links used below.

In [ ]:
# If needed, uncomment this line once:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.4f}".format

# Freeze this for a submitted assignment, e.g. "2026-09-14".
AS_OF_DATE = "2026-09-14"
TRAIN_START = "2009-01-01"   # target-range series is consistently available after Dec. 2008
RANDOM_STATE = 42

## 2. Download and verify the historical data

A **yield** is the annualized return investors demand to lend to the U.S. government. A 3-month Treasury yield reflects very short-term conditions; a 30-year yield also embeds long-run inflation, growth, and risk expectations.

The Fed does **not** set Treasury yields. It sets a target range for the overnight federal-funds rate. Treasury yields respond to expectations about future Fed policy **and** many other forces.

In [ ]:
SERIES = {
    "DGS3MO": "3-month Treasury",
    "DGS6MO": "6-month Treasury",
    "DGS10": "10-year Treasury",
    "DGS30": "30-year Treasury",
    "DFEDTARU": "Fed target upper bound",
    "DFEDTARL": "Fed target lower bound",
}

def download_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    frame = pd.read_csv(url, parse_dates=["observation_date"])
    return frame.rename(columns={"observation_date": "date"})

frames = [download_fred(s) for s in SERIES]
daily = reduce(lambda left, right: left.merge(right, on="date", how="outer"), frames)
daily = daily.sort_values("date").set_index("date").loc[:AS_OF_DATE]

for column in SERIES:
    daily[column] = pd.to_numeric(daily[column], errors="coerce")

verification = pd.DataFrame({
    "description": pd.Series(SERIES),
    "first_date": [daily[s].first_valid_index() for s in SERIES],
    "latest_date": [daily[s].last_valid_index() for s in SERIES],
    "latest_value_percent": [daily[s].dropna().iloc[-1] for s in SERIES],
})
verification

In [ ]:
yield_cols = ["DGS3MO", "DGS6MO", "DGS10", "DGS30"]

latest_yield_date = daily[yield_cols].dropna().index.max()
latest_curve = daily.loc[latest_yield_date, yield_cols]
latest_target_date = daily["DFEDTARU"].dropna().index.max()
latest_target_upper = daily.loc[latest_target_date, "DFEDTARU"]

print(f"Latest complete Treasury curve: {latest_yield_date.date()}")
print(latest_curve.rename(SERIES).to_string())
print(f"\nFed target upper bound on {latest_target_date.date()}: {latest_target_upper:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
daily.loc["2021":, yield_cols + ["DFEDTARU"]].plot(ax=ax, linewidth=1.5)
ax.set(title="Treasury yields and the Fed target upper bound", ylabel="Percent", xlabel="")
ax.legend([SERIES[c] for c in yield_cols + ["DFEDTARU"]], ncol=2)
plt.tight_layout()

## 3. Define the event and the time unit

We use a **discrete-time hazard model**. Each row is one calendar month.

- `hike = 1` if the Fed target upper bound rose during that month.
- `hike = 0` otherwise.
- Predictors come from the **end of the previous month**, so the model never uses a yield observed after a historical decision to “predict” that same decision.

For a future month \(t\), the hazard is

$$
h_t = P(\text{hike in month }t \mid \text{no earlier hike in this forecast window}, X_t).
$$

We estimate it with regularized logistic regression:

$$
h_t = \frac{1}{1 + e^{-(\beta_0 + X_t\beta)}}.
$$

This is a standard discrete-time event-history model. “Hazard” means a **conditional chance during the current interval**, not that the chance automatically rises merely because time passes.

In [ ]:
# Last available observation in each month.
monthly_yields = daily[yield_cols].dropna().resample("ME").last()
monthly_target = daily[["DFEDTARU"]].dropna().resample("ME").last()
monthly = monthly_yields.join(monthly_target).loc[TRAIN_START:].dropna()

# Do not train on the incomplete forecast month.
forecast_month = pd.Timestamp(AS_OF_DATE).to_period("M")
monthly = monthly[monthly.index.to_period("M") < forecast_month].copy()

# A positive month-to-month change in the target upper bound marks a hike month.
monthly["hike"] = (monthly["DFEDTARU"].diff() > 0).astype(int)

# Lag all raw predictors one month to prevent look-ahead bias.
for col in yield_cols + ["DFEDTARU"]:
    monthly[f"lag_{col}"] = monthly[col].shift(1)

monthly = monthly.dropna().copy()

print(f"Training months: {len(monthly)}")
print(f"Historical hike months: {monthly['hike'].sum()}")
print(f"Historical monthly base rate: {monthly['hike'].mean():.1%}")
monthly.loc[monthly["hike"].eq(1), ["DFEDTARU", "hike"]].tail(15)

## 4. Turn the four yields into interpretable curve features

Using four raw yields together creates heavy overlap because they often move together. We retain information from all four maturities while expressing it as economically readable gaps:

- **Short-rate gap:** 3-month yield minus the Fed target upper bound. A positive value says very short Treasuries trade above today's policy ceiling.
- **Near-term slope:** 6-month minus 3-month yield. A positive value says rates are higher a little farther ahead.
- **Curve slope:** 10-year minus 3-month yield. This captures the broad short-to-long shape.
- **Long-end slope:** 30-year minus 10-year yield. This isolates the far end of the curve.

These are associations, not proof that the yield curve causes Fed decisions.

In [ ]:
monthly["short_gap"] = monthly["lag_DGS3MO"] - monthly["lag_DFEDTARU"]
monthly["near_slope"] = monthly["lag_DGS6MO"] - monthly["lag_DGS3MO"]
monthly["curve_slope"] = monthly["lag_DGS10"] - monthly["lag_DGS3MO"]
monthly["long_slope"] = monthly["lag_DGS30"] - monthly["lag_DGS10"]

features = ["short_gap", "near_slope", "curve_slope", "long_slope"]
monthly.groupby("hike")[features].mean().rename(index={0: "No hike", 1: "Hike"})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for feature, ax in zip(features, axes.flat):
    sns.boxplot(data=monthly, x="hike", y=feature, ax=ax)
    ax.set_xticklabels(["No hike", "Hike"])
    ax.set_xlabel("")
fig.suptitle("Previous-month yield-curve features before hike and no-hike months", y=1.02)
plt.tight_layout()

## 5. Select regularization without peeking forward

Hikes are rare and tend to arrive in cycles, so an unregularized model can become unstable. `C` controls regularization: smaller `C` shrinks coefficients more strongly.

We choose `C` using expanding-window time-series validation. Each validation fold occurs **after** its training data. Log loss rewards useful probabilities and strongly penalizes confident errors.

In [ ]:
X = monthly[features]
y = monthly["hike"]

c_grid = [0.01, 0.03, 0.10, 0.30, 1.00, 3.00]
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for c in c_grid:
    fold_losses = []
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        if y_train.nunique() < 2:
            continue
        candidate = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=c, max_iter=10_000, random_state=RANDOM_STATE),
        )
        candidate.fit(X_train, y_train)
        probability = candidate.predict_proba(X_test)[:, 1]
        fold_losses.append(log_loss(y_test, probability, labels=[0, 1]))
    cv_rows.append({"C": c, "mean_time_series_log_loss": np.mean(fold_losses)})

cv_results = pd.DataFrame(cv_rows).sort_values("mean_time_series_log_loss")
best_c = cv_results.iloc[0]["C"]
display(cv_results)
print(f"Selected C: {best_c}")

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=best_c, max_iter=10_000, random_state=RANDOM_STATE),
)
model.fit(X, y)

fitted_probability = model.predict_proba(X)[:, 1]
diagnostics = pd.Series({
    "ROC AUC (ranking; higher is better)": roc_auc_score(y, fitted_probability),
    "Brier score (probability error; lower is better)": brier_score_loss(y, fitted_probability),
    "Historical base rate": y.mean(),
})
diagnostics

The in-sample diagnostics are descriptive, not a guarantee of future performance. Hike cycles create regime changes, and a Treasury-only model omits inflation, employment, Fed communication, and fed-funds futures. The professional comparison later is therefore essential.

## 6. Create the September–December 2026 forecast

At forecast time, we transform the latest complete curve exactly as we transformed historical curves. Since future October and December yields are unknown, the base case holds the latest curve constant. That is a transparent **constant-information assumption**, not a claim that yields will actually remain unchanged.

November has no scheduled FOMC meeting, so its scheduled-meeting hazard is set to zero. The model excludes unscheduled emergency actions.

In [ ]:
current_features = pd.DataFrame([{
    "short_gap": latest_curve["DGS3MO"] - latest_target_upper,
    "near_slope": latest_curve["DGS6MO"] - latest_curve["DGS3MO"],
    "curve_slope": latest_curve["DGS10"] - latest_curve["DGS3MO"],
    "long_slope": latest_curve["DGS30"] - latest_curve["DGS10"],
}])

display(current_features.T.rename(columns={0: "percentage-point difference"}))

base_hazard = model.predict_proba(current_features[features])[:, 1][0]

forecast = pd.DataFrame({
    "month": ["September 2026", "October 2026", "November 2026", "December 2026"],
    "meeting": ["Sep. 15–16", "Oct. 27–28", "No scheduled meeting", "Dec. 8–9"],
    "scheduled_meeting": [True, True, False, True],
})
forecast["conditional_hazard"] = np.where(forecast["scheduled_meeting"], base_hazard, 0.0)

# Probability that the FIRST hike in this forecast window occurs in each month.
survival = 1.0
first_hike = []
survival_after = []
for hazard in forecast["conditional_hazard"]:
    first_hike.append(survival * hazard)
    survival *= (1 - hazard)
    survival_after.append(survival)

forecast["first_hike_probability"] = first_hike
forecast["survival_after_month"] = survival_after

forecast_display = forecast.copy()
for col in ["conditional_hazard", "first_hike_probability", "survival_after_month"]:
    forecast_display[col] = forecast_display[col].map(lambda value: f"{value:.1%}")
display(forecast_display)
print(f"Probability of no hike through December under this model: {survival:.1%}")

### How to read the two probability columns

Suppose the model reports a 48% conditional hazard at each scheduled meeting:

- **September conditional hazard:** chance of a September hike given no earlier hike in this forecast window.
- **October conditional hazard:** chance of an October hike **if September did not produce a hike**.
- **October first-hike probability:** chance that September produces no hike *and then* October produces the first hike: $$(1-h_{Sep})h_{Oct}$$.

The hazard and survival are related, but they are not simple complements for one month. After several periods,

$$
S_t = \prod_{j \le t}(1-h_j), \qquad
P(T=t)=S_{t-1}h_t.
$$

In [ ]:
plot_data = forecast.melt(
    id_vars="month",
    value_vars=["conditional_hazard", "first_hike_probability"],
    var_name="probability_type",
    value_name="probability",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_data, x="month", y="probability", hue="probability_type", ax=ax)
ax.set(title="Treasury-curve hazard model", xlabel="", ylabel="Probability", ylim=(0, 1))
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()

## 7. Bootstrap uncertainty interval

One fitted number can look more precise than it is. The block bootstrap resamples consecutive 12-month chunks, preserving some of the clustering in hiking cycles. It refits the entire model and produces an empirical interval for the current conditional hazard.

This interval measures **sampling/model uncertainty under this specification**. It does not capture omitted variables or a brand-new economic regime.

In [ ]:
def moving_block_sample(frame, block_length, rng):
    n = len(frame)
    starts = rng.integers(0, n - block_length + 1, size=int(np.ceil(n / block_length)))
    positions = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]
    return frame.iloc[positions]

rng = np.random.default_rng(RANDOM_STATE)
bootstrap_probabilities = []

for _ in range(1000):
    sample = moving_block_sample(monthly, block_length=12, rng=rng)
    if sample["hike"].nunique() < 2:
        continue
    boot_model = clone(model)
    boot_model.fit(sample[features], sample["hike"])
    bootstrap_probabilities.append(
        boot_model.predict_proba(current_features[features])[:, 1][0]
    )

interval = np.quantile(bootstrap_probabilities, [0.025, 0.50, 0.975])
print(f"Point estimate: {base_hazard:.1%}")
print(f"Bootstrap median: {interval[1]:.1%}")
print(f"95% bootstrap interval: {interval[0]:.1%} to {interval[2]:.1%}")

## 8. Validation against professional evidence

The comparison must be **as of the same date** as the model. On September 14, 2026:

- [Reuters reported](https://www.reuters.com/business/goldman-sachs-now-expects-fed-hike-rates-september-2026-09-14/) that Goldman Sachs, J.P. Morgan, HSBC, and Deutsche Bank expected a 25-basis-point September hike. It also reported roughly **90%** September odds from CME FedWatch and another increase expected in December.
- [The Wall Street Journal reported](https://www.wsj.com/livecoverage/stock-market-cpi-inflation-09-11-2026/card/goldman-sachs-analysts-now-expect-a-september-rate-hike-B4KhrjQPwJPqiTkWqJqm) that Goldman Sachs viewed a September hike as likely, while further hikes were possible but not its baseline.
- The [official Fed calendar](https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm) confirms meetings in September, October, and December—and no November meeting.

Run the next cell after the model, then compare its September estimate with the professional benchmark.

In [ ]:
professional_comparison = pd.DataFrame({
    "source_or_method": [
        "Treasury-only hazard model",
        "CME FedWatch reported by Reuters (Sep. 14, 2026)",
        "Goldman/J.P. Morgan/HSBC/Deutsche Bank reported consensus",
    ],
    "september_2026_view": [
        f"{base_hazard:.1%} conditional probability",
        "About 90% probability of a 25 bp hike",
        "25 bp hike expected",
    ],
    "interpretation": [
        "Historical association using four Treasury maturities only",
        "Market-implied estimate from fed-funds futures",
        "Economists' forecast using inflation, activity, policy signals, and markets",
    ],
})
professional_comparison

## 9. Analyst conclusion

Use the model output, but do **not** describe it as “the market's probability.” A defensible conclusion is:

> The discrete-time hazard model estimates the conditional probability of a hike from historical relationships between the prior yield curve and Fed target changes. As of the latest Treasury observation, the model assigns the displayed conditional hazard to each scheduled meeting under a constant-curve assumption; November is 0% because no routine FOMC meeting is scheduled. The first-hike probabilities decline across later meetings because the event must not have occurred earlier. Professional evidence is materially more hawkish for September, with fed-funds futures near 90% and several major banks expecting a 25-basis-point move. The gap indicates omitted real-time information—especially inflation news and Fed communication—and shows why Treasury yields alone should be treated as a deliberately limited classroom model rather than a trading-grade forecast.

### Limitations to report

1. **Small number of hike events.** Since 2009, hikes occur in a few clustered cycles.
2. **Treasury yields are indirect.** Longer maturities reflect growth, inflation, term premiums, and supply—not only Fed policy.
3. **Constant future curve.** October and December use the latest observed curve because their future curves are unknowable today.
4. **No macro variables.** The assignment-restricted model omits inflation, employment, and Fed communications.
5. **First-event framework.** Survival probabilities answer when the first hike occurs. They do not model a second hike after the first.
6. **Calendar structure.** November's 0% refers to scheduled meetings and excludes emergency intermeeting action.
7. **Forecasts change.** Rerunning with new data can materially change the result; retain the as-of date.

## 10. Save data for Part 2

These files can feed the later Pudding-style GitHub Pages visualization. They contain no proprietary data.

In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

forecast.to_csv(output_dir / "hazard_forecast_2026.csv", index=False)
monthly.reset_index().to_csv(output_dir / "historical_model_data.csv", index=False)
daily[yield_cols + ["DFEDTARU", "DFEDTARL"]].reset_index().to_csv(
    output_dir / "official_daily_rates.csv", index=False
)

print("Saved:")
for path in output_dir.iterdir():
    print(" -", path)